<a href="https://colab.research.google.com/github/VitorBZS/PLN-A940-M-D.S.M.-297-20262/blob/main/Exerc%C3%ADcio_Pr%C3%A1tico_Individual_Aula_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q spacy pandas
!python -m spacy download pt_core_news_sm -q

import spacy
import pandas as pd

nlp = spacy.load("pt_core_news_sm")

print("Ambiente configurado com sucesso.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 89.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Ambiente configurado com sucesso.


In [3]:
lexico_positivo = {
    "ótimo", "excelente", "adorar", "maravilhoso", "perfeito", "recomendar",
    "satisfeito", "rápido", "eficiente", "atencioso", "gentil", "confiável",
    "espaçoso", "iluminado", "tranquilo", "seguro", "bom"
}

lexico_negativo = {
    "péssimo", "horrível", "detestar", "decepcionado", "lento", "descaso",
    "problema", "sujo", "escuro", "barulhento", "caro", "insatisfeito",
    "atrasado", "ruim", "complicado", "incomodar"
}

##1. Classificação de cada comentário

In [9]:
def polaridade_simples(texto):
  doc = nlp(texto)
  positivas = sum(1 for token in doc if token.lemma_.lower() in lexico_positivo)
  negativas = sum(1 for token in doc if token.lemma_.lower() in lexico_negativo)

  if positivas > negativas:
      classificacao = "Positivo"
  elif negativas > positivas:
      classificacao = "Negativo"
  else:
      classificacao = "Neutro"

  return classificacao, positivas, negativas

In [10]:
# Espaço do aluno — escreva seu código aqui
corpus_exercicio = [
    "O síndico é muito prestativo e sempre resolve tudo rápido.",
    "A área de lazer não estava limpa quando visitei.",
    "O condomínio é caro, mas a estrutura compensa.",
    "Nunca vi um prédio tão mal administrado.",
]

for i, texto in enumerate(corpus_exercicio, 1):
  classificacao, pos, neg = polaridade_simples(texto)
  print(f"Review {i}: {classificacao} (positivas = {pos}, negativas = {neg})")
  print(f" \"{texto}\"")

Review 1: Positivo (positivas = 1, negativas = 0)
 "O síndico é muito prestativo e sempre resolve tudo rápido."
Review 2: Neutro (positivas = 0, negativas = 0)
 "A área de lazer não estava limpa quando visitei."
Review 3: Negativo (positivas = 0, negativas = 1)
 "O condomínio é caro, mas a estrutura compensa."
Review 4: Neutro (positivas = 0, negativas = 0)
 "Nunca vi um prédio tão mal administrado."


##2. Investigação dos resultados inesperados

In [11]:
doc_review1 = nlp(corpus_exercicio[2])
for token in doc_review1:
  print(f"{token.text:15} -> lema: {token.lemma_}")

O               -> lema: o
condomínio      -> lema: condomínio
é               -> lema: ser
caro            -> lema: caro
,               -> lema: ,
mas             -> lema: mas
a               -> lema: o
estrutura       -> lema: estrutura
compensa        -> lema: compenso
.               -> lema: .


In [12]:
doc_review1 = nlp(corpus_exercicio[3])
for token in doc_review1:
  print(f"{token.text:15} -> lema: {token.lemma_}")

Nunca           -> lema: nunca
vi              -> lema: ver
um              -> lema: um
prédio          -> lema: prédio
tão             -> lema: tão
mal             -> lema: mal
administrado    -> lema: administrado
.               -> lema: .


##3. Ajuste do léxico dos resultados inesperado


In [15]:
lexico_negativo_v2 = lexico_negativo | {"mal"}
lexico_positivo_v2 = lexico_positivo | {"compenso"}

def polaridade_v2(texto):
  doc = nlp(texto)
  positivas = sum(1 for token in doc if token.lemma_.lower() in lexico_positivo_v2)
  negativas = sum(1 for token in doc if token.lemma_.lower() in lexico_negativo_v2)
  if positivas > negativas:
      return "Positivo", positivas, negativas
  elif negativas > positivas:
      return "Negativo", positivas, negativas
  return "Neutro", positivas, negativas

print("Antes:", polaridade_simples(corpus_exercicio[2]))
print("Depois:", polaridade_v2(corpus_exercicio[2]))
print()
print("Antes:", polaridade_simples(corpus_exercicio[3]))
print("Depois:", polaridade_v2(corpus_exercicio[3]))


Antes: ('Negativo', 0, 1)
Depois: ('Neutro', 1, 1)

Antes: ('Neutro', 0, 0)
Depois: ('Negativo', 0, 1)
